In [ ]:
# Importing required libraries

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from surprise import Reader, Dataset, SVD, accuracy
from surprise.model_selection import train_test_split

## Pre-Processing

In [ ]:
# Adjusting row column settings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)
pd.set_option('display.expand_frame_repr', False)

In [ ]:
# Loading the dataset

movie_df = pd.read_csv('movie.csv')
rating_df = pd.read_csv('rating.csv')

In [ ]:
# Checking No. of Unique Values in the Datasets

movie_df.nunique()

movieId    27278
title      27262
genres      1342
dtype: int64

In [ ]:
rating_df.nunique()

userId         138493
movieId         26744
rating             10
timestamp    15351121
dtype: int64

In [ ]:
# Reducing the rating dataset

unique_users = rating_df['userId'].unique()
half_users = np.random.choice(unique_users, size=round(0.5*(len(unique_users))), replace=False)
half_rating_df = rating_df[rating_df['userId'].isin(half_users)]
print("Original DataFrame shape:", rating_df.shape)
print("New DataFrame shape after removing half the users:", half_rating_df.shape)

Original DataFrame shape: (20000263, 4)
New DataFrame shape after removing half the users: (9950537, 4)


In [ ]:
half_rating_df.head()

,userId,movieId,rating,timestamp
236,3,1,4.0,1999-12-11 13:36:47
237,3,24,3.0,1999-12-14 12:54:08
238,3,32,4.0,1999-12-11 13:14:07
239,3,50,5.0,1999-12-11 13:13:38
240,3,160,3.0,1999-12-14 12:54:08


In [ ]:
# Merging movie and rating datasets

df = movie_df.merge(half_rating_df, how="left", on="movieId")

In [ ]:
# Preliminary examination of the data set

def check_df(dataframe, head=5):
    print('##################### Shape #####################')
    print(dataframe.shape)
    print('##################### Types #####################')
    print(dataframe.dtypes)
    print('##################### Head #####################')
    print(dataframe.head(head))
    print('##################### Tail #####################')
    print(dataframe.tail(head))
    print('##################### NA #####################')
    print(dataframe.isnull().sum())
    print('##################### Quantiles #####################')
    print(dataframe['rating'].describe([0, 0.05, 0.50, 0.95, 0.99, 1]))

check_df(df)

##################### Shape #####################
(9953889, 6)
##################### Types #####################
movieId        int64
title         object
genres        object
userId       float64
rating       float64
timestamp     object
dtype: object
##################### Head #####################
   movieId             title                                       genres  userId  rating            timestamp
0        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy     3.0     4.0  1999-12-11 13:36:47
1        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy     6.0     5.0  1997-03-13 17:50:52
2        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy    10.0     4.0  1999-11-25 02:44:47
3        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy    14.0     4.5  2008-10-29 20:13:59
4        1  Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy    19.0     5.0  1997-02-05 21:03:48
##################### Tail #####

## Creating a Pivot Table

In [ ]:
# Step-1: Calculate the total number of ratings cast for each movie.

rating_counts = pd.DataFrame(df["title"].value_counts())
rating_counts.head(10)

,title
Pulp Fiction (1994),33624
Forrest Gump (1994),32953
"Silence of the Lambs, The (1991)",31663
"Shawshank Redemption, The (1994)",31514
Jurassic Park (1993),29743
Star Wars: Episode IV - A New Hope (1977),27249
Braveheart (1995),26779
Terminator 2: Judgment Day (1991),26099
"Matrix, The (1999)",25608
Schindler's List (1993),25080


In [ ]:
rating_counts.shape

(27262, 1)

In [ ]:
# Step-2: Store the names of films with less than 500 total votes in rare_movies and remove them from the dataset.

rare_movies = rating_counts[rating_counts['title'] <= 500].index
common_movies = df[~df["title"].isin(rare_movies)]
common_movies.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.0,4.0,1999-12-11 13:36:47
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,6.0,5.0,1997-03-13 17:50:52
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,10.0,4.0,1999-11-25 02:44:47
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,14.0,4.5,2008-10-29 20:13:59
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,19.0,5.0,1997-02-05 21:03:48


In [ ]:
common_movies.shape

(8831534, 6)

In [ ]:
len(pd.unique(common_movies['title']))

3129

In [ ]:
# Step-3: Create a pivot table for dataframe with userIDs in index, movieIds in columns and ratings as values.

user_movie_df = common_movies.pivot_table(index=["userId"], columns=["movieId"], values="rating")
user_movie_df.head()

movieId  1       2       3       4       5       6       7       8       9       10      11      12      13      14      15      16      17      18      19      20      21      22      23      24      25      26      27      28      29      30      31      32      34      35      36      39      41      42      43      44      45      46      47      48      50      52      57      58      60      61      62      63      64      65      66      68      69      70      71      72      73      74      76      78      79      81      82      85      86      88      89      92      93      94      95      97      100     101     102     103     104     105     107     110     111     112     113     117     118     122     123     125     126     132     135     140     141     144     145     147     149     150     151     153     154     155     156     157     158     159     160     161     162     163     164     165     168     169     170     171     172     173     174     175     176     177     179     180     181     185     186     188     191     193     194     195     196     198     199     203     204     205     207     208     213     215     216     217     218     222     223     224     225     227     229     230     231     232     233     234     235     236     237     239     242     246     247     248     249     252     253     255     256     257     258     259     260     261     262     265     266     267     270     271     272     273     274     275     276     277     278     280     281     282     288     289     290     292     293     296     299     300     302     303     305     306     307     308     312     313     314     315     316     317     318     319     322     325     326     327     328     329     330     332     333     334     337     338     339     340     342     344     345     346     347     348     349     350     351     352     353     354     355     356     357     358     360     361     362     364     365     366     367     368     369     370     371     372     373     374     376     377     378     379     380     381     382     383     393     405     407     410     412     413     415     416     417     419     420     421     422     423     426     427     428     429     431     432     433     434     435     436     437     438     440     441     442     444     445     446     448     450     452     454     455     457     458     459     464     466     468     469     471     473     474     475     477     479     480     481     482     485     489     490     491     492     493     494     497     500     501     502     504     505     506     507     508     509     511     512     514     515     516     517     518     519     520     521     522     524     527     529     531     532     533     534     535     537     538     539     540     541     542     543     544     546     548     549     550     551     552     553     555     556     558     562     569     574     575     581     585     586     587     588     589     590     592     593     594     595     596     597     599     605     606     608     609     610     611     612     613     616     627     628     631     635     637     639     640     647     648     650     653     656     661     662     663     665     667     671     673     674     678     680     688     691     694     697     700     704     707     708     709     710     711     714     719     720     724     725     728     733     735     736     737     741     742     743     745     747     748     750     761     762     765     766     778     780     781     782     783     784     785     786     788     798     799     800     801     802     804     805     809     810     818     828     829     830     832     833     836     837     838     839     842     848     849     851     852     858     861     866     879     880     881     886     891     892     898    

In [ ]:
user_movie_df.shape

(69246, 3133)

## Determining the movies watched by a selected user

In [ ]:
# Step-1: Choose a random user id.

target_user_id = 6

In [ ]:
# Step-2: Create a new dataframe named random_user_df consisting of observation units of the selected user.

target_user_df = user_movie_df[user_movie_df.index == target_user_id]
target_user_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,34,35,36,39,41,42,43,44,45,46,47,48,50,52,57,58,60,61,62,63,64,65,66,68,69,70,71,72,73,74,76,78,79,81,82,85,86,88,89,92,93,94,95,97,100,101,102,103,104,105,107,110,111,112,113,117,118,122,123,125,126,132,135,140,141,144,145,147,149,150,151,153,154,155,156,157,158,159,160,161,162,163,164,165,168,169,170,171,172,173,174,175,176,177,179,180,181,185,186,188,191,193,194,195,196,198,199,203,204,205,207,208,213,215,216,217,218,222,223,224,225,227,229,230,231,232,233,234,235,236,237,239,242,246,247,248,249,252,253,255,256,257,258,259,260,261,262,265,266,267,270,271,272,273,274,275,276,277,278,280,281,282,288,289,290,292,293,296,299,300,302,303,305,306,307,308,312,313,314,315,316,317,318,319,322,325,326,327,328,329,330,332,333,334,337,338,339,340,342,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,360,361,362,364,365,366,367,368,369,370,371,372,373,374,376,377,378,379,380,381,382,383,393,405,407,410,412,413,415,416,417,419,420,421,422,423,426,427,428,429,431,432,433,434,435,436,437,438,440,441,442,444,445,446,448,450,452,454,455,457,458,459,464,466,468,469,471,473,474,475,477,479,480,481,482,485,489,490,491,492,493,494,497,500,501,502,504,505,506,507,508,509,511,512,514,515,516,517,518,519,520,521,522,524,527,529,531,532,533,534,535,537,538,539,540,541,542,543,544,546,548,549,550,551,552,553,555,556,558,562,569,574,575,581,585,586,587,588,589,590,592,593,594,595,596,597,599,605,606,608,609,610,611,612,613,616,627,628,631,635,637,639,640,647,648,650,653,656,661,662,663,665,667,671,673,674,678,680,688,691,694,697,700,704,707,708,709,710,711,714,719,720,724,725,728,733,735,736,737,741,742,743,745,747,748,750,761,762,765,766,778,780,781,782,783,784,785,786,788,798,799,800,801,802,804,805,809,810,818,828,829,830,832,833,836,837,838,839,842,848,849,851,852,858,861,866,879,880,881,886,891,892,898,899,900,901,902,903,904,905,906,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,926,928,930,931,932,933,934,936,938,940,942,943,944,945,946,947,948,949,950,951,952,953,954,955,965,968,969,971,986,991,994,996,999,1003,1004,1005,1006,1007,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1041,1042,1043,1046,1047,1049,1050,1051,1057,1059,1060,1061,1064,1066,1073,1077,1078,1079,1080,1081,1082,1083,1084,1085,1086,1088,1089,1090,1091,1092,1093,1094,1095,1096,1097,1099,1100,1101,1103,1104,1111,1113,1120,1124,1125,1126,1127,1128,1129,1130,1131,1132,1135,1136,1147,1148,1150,1161,1171,1172,1173,1175,1176,1177,1178,1179,1183,1184,1185,1186,1187,1188,1189,1190,1191,1192,1193,1194,1196,1197,1198,1199,1200,1201,1202,1203,1204,1206,1207,1208,1209,1210,1211,1212,1213,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1226,1227,1228,1230,1231,1232,1233,1234,1235,1236,1237,1238,1240,1241,1242,1243,1244,1245,1246,1247,1248,1249,1250,1251,1252,1253,1254,1255,1256,1257,1258,1259,1260,1261,1262,1263,1264,1265,1266,1267,1268,1269,1270,1271,1272,1273,1274,1275,1276,1277,1278,1279,1280,1281,1282,1283,1284,1285,1286,1287,1288,1289,1290,1291,1292,1293,1295,1296,1297,1298,1299,1300,1301,1302,1303,1304,1305,1306,1307,1320,1321,1327,1333,1334,1339,1340,1342,1343,1344,1345,1346,1347,1348,1350,1353,1354,1356,1357,1358,1359,1361,1363,1365,1366,1367,1370,1371,1372,1373,1374,1375,1376,1377,1378,1379,1380,1381,1385,1387,1388,1389,1390,1391,1392,1393,1394,1395,1396,1399,1401,1405,1407,1408,1409,1411,1414,1416,1419,1422,1425,1429,1431,1432,1438,1441,1445,1446,1449,1453,1457,1459,1461,1464,1466,1468,1474,1476,1479,1480,1483,1485,1487,1488,1498,1499,1500,1503,1513,1515,1517,1518,1527,1537,1541,1542,1544,1552,1554,1556,1562,1566,1569,1573,1580,1584,1586,1587,1588,1589,1590,1591,1592,1593,1594,1597,1603,1608,1610,1611,1614,1615,1616,1617,1619,1620,1625,1627,1633,1635,1639,1641,1643,1644,1645,1648,1649,1653,1658,1660,1663,1665,1667,1672,1673,1674,1676,1678,1680,1681,1682

In [ ]:
target_user_df.shape

(1, 3133)

In [ ]:
# Step-3: Assign the films voted by the selected user to a list called movies_watched.

movies_watched_id = target_user_df.columns[target_user_df.notna().any()].to_list()
movies_watched_id

[1,
 3,
 7,
 17,
 52,
 62,
 135,
 140,
 141,
 260,
 494,
 628,
 648,
 653,
 708,
 719,
 733,
 736,
 743,
 762,
 780,
 788,
 802,
 1073]

In [ ]:
# Making a function to convert Movie Ids to Movie Titles

def convert_id_to_title (id_list):
    movie_titles = []
    for r in id_list:
            df1 = movie_df[movie_df['movieId']==r]
            title = df1['title'].values[0]
            movie_titles.append(title)
    return movie_titles

convert_id_to_title(movies_watched_id)

['Toy Story (1995)',
 'Grumpier Old Men (1995)',
 'Sabrina (1995)',
 'Sense and Sensibility (1995)',
 'Mighty Aphrodite (1995)',
 "Mr. Holland's Opus (1995)",
 'Down Periscope (1996)',
 'Up Close and Personal (1996)',
 'Birdcage, The (1996)',
 'Star Wars: Episode IV - A New Hope (1977)',
 'Executive Decision (1996)',
 'Primal Fear (1996)',
 'Mission: Impossible (1996)',
 'Dragonheart (1996)',
 'Truth About Cats & Dogs, The (1996)',
 'Multiplicity (1996)',
 'Rock, The (1996)',
 'Twister (1996)',
 'Spy Hard (1996)',
 'Striptease (1996)',
 'Independence Day (a.k.a. ID4) (1996)',
 'Nutty Professor, The (1996)',
 'Phenomenon (1996)',
 'Willy Wonka & the Chocolate Factory (1971)']

In [ ]:
len(movies_watched_id)

24

# Method 1 - Content-Based Filtering

In [ ]:
# Step-1: Find the id of the movie that the user has most recently rated 5

movie_id = half_rating_df[(half_rating_df["userId"] == target_user_id) & (half_rating_df["rating"] == 5.0)].sort_values(by="timestamp", ascending=False)["movieId"][0:1].values[0]

In [ ]:
# Step-2: Find the title of this movie

movie_title = movie_df[movie_df["movieId"] == movie_id]["title"].values[0]
movie_title

'Mighty Aphrodite (1995)'

In [ ]:
# Step-3: Create a dataframe for this movie

best_movie_df = movie_df[movie_df["movieId"] == movie_id]
best_movie_df

,movieId,title,genres
51,52,Mighty Aphrodite (1995),Comedy|Drama|Romance


In [ ]:
# Step 4: Get user's preferred genres based on their highly-rated movies

def get_user_preferred_genres(user_id, min_rating=4.0):
    # Get the movies rated by the user
    user_data = df[df['userId'] == user_id]

    # Filter for movies with a rating above the given threshold
    top_rated_movies = user_data[user_data['rating'] >= min_rating]

    # Get the genres of the top-rated movies
    preferred_genres = top_rated_movies['genres'].str.split('|').explode().unique()

    return preferred_genres

get_user_preferred_genres(target_user_id)

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Sci-Fi', 'Thriller', 'Crime',
       'Mystery'], dtype=object)

In [ ]:
# Step 5: Filter unwatched movies based on preferred genres

def get_filtered_movies(user_id, preferred_genres):
    # Get the list of movies the user has already watched
    user_data = df[df['userId'] == user_id]
    user_watched_movies = user_data['movieId'].tolist()

    # Filter out movies the user has already watched
    unwatched_movies = movie_df[~movie_df['movieId'].isin(user_watched_movies)]

    # Further filter movies by matching genres
    filtered_movies = unwatched_movies[unwatched_movies['genres'].apply(
        lambda g: any(genre in g.split('|') for genre in preferred_genres)
    )]

    #Merge it with the best_movie to apply cosine similarity
    filtered_movies = pd.concat([filtered_movies, best_movie_df], ignore_index = True)
    filtered_movies.reset_index()

    return filtered_movies


a = get_filtered_movies(target_user_id, get_user_preferred_genres(target_user_id))

print("Filtered Movies Preview (head):")
print(a.head())

print("\nFiltered Movies Preview (tail):")
print(a.tail())

Filtered Movies Preview (head):
   movieId                               title                      genres
0        2                      Jumanji (1995)  Adventure|Children|Fantasy
1        4            Waiting to Exhale (1995)        Comedy|Drama|Romance
2        5  Father of the Bride Part II (1995)                      Comedy
3        6                         Heat (1995)       Action|Crime|Thriller
4        8                 Tom and Huck (1995)          Adventure|Children

Filtered Movies Preview (tail):
       movieId                          title                    genres
23960   131254   Kein Bund für's Leben (2007)                    Comedy
23961   131256  Feuer, Eis & Dosenbier (2002)                    Comedy
23962   131258             The Pirates (2014)                 Adventure
23963   131262               Innocence (2014)  Adventure|Fantasy|Horror
23964       52        Mighty Aphrodite (1995)      Comedy|Drama|Romance


In [ ]:
# Step 6: Make a function to apply content-based filtering

def get_content_based_recommendations_for_user(user_id, top_n=5, min_rating=4.0):
    # Get user's preferred genres
    preferred_genres = get_user_preferred_genres(user_id, min_rating)

    # Get filtered movies (unwatched by the user and matching preferred genres)
    filtered_df = get_filtered_movies(user_id, preferred_genres)

    # Create a new content DataFrame with only filtered unwatched movies
    content_df = filtered_df[['movieId', 'genres']]
    content_df['Content'] = content_df.fillna('').astype(str).agg(' '.join, axis=1)

    # Apply TF-IDF to this reduced set of movies
    tfidf_vectorizer = TfidfVectorizer()
    content_matrix = tfidf_vectorizer.fit_transform(content_df['Content'])

    # Cosine similarity calculation on the reduced set
    content_similarity = linear_kernel(content_matrix, content_matrix)

    # Get user's latest top-rated movie
    latest_top_rated_movie = movie_id

    # Get index of the latest top-rated movie in the reduced set
    movie_index = content_df[content_df['movieId'] == latest_top_rated_movie].index[0]

    # Get similarity scores for that movie
    similarity_scores = content_similarity[movie_index]

    # Sort and get top N similar movie indices (excluding the movie itself)
    similar_indices = similarity_scores.argsort()[::-1][1:top_n + 1]

    # Get the corresponding movie id of the recommendations
    recommendations_id = content_df.loc[similar_indices, 'movieId'].values

    return recommendations_id

In [ ]:
#Example Usage
cbf_recommendations = get_content_based_recommendations_for_user(user_id=target_user_id)
cbf_movieId = cbf_recommendations.tolist()

In [ ]:
print(cbf_movieId)

[4, 5, 103920, 103539, 103606]


In [ ]:
# Making a function to convert Movie Ids to Movie Titles & Genres

def convert_id_to_title_and_genre (id_list):
    recommendations = []
    for r in id_list:
            df1 = movie_df[movie_df['movieId']==r]
            title = df1['title'].values[0]
            genre = df1['genres'].values[0]
            recommendations.append((title, genre))

    for movie, genre in recommendations:
        print(f"Movie: {movie}, Genre: {genre}")

convert_id_to_title_and_genre(cbf_movieId)

Movie: Waiting to Exhale (1995), Genre: Comedy|Drama|Romance
Movie: Father of the Bride Part II (1995), Genre: Comedy
Movie: All Together, The (2007), Genre: Comedy|Drama|Romance
Movie: The Spectacular Now (2013), Genre: Comedy|Drama|Romance
Movie: Stuck in Love (2012), Genre: Comedy|Drama|Romance


# Method 2 - Collaborative Filtering

## 2.1 Correlation Method - Pearson's Correlation Coefficient

### Accessing Data and Ids of Other Users Watching the Same Films

In [ ]:
# Step-1: Create a new dataframe containing only the films watched by the selected user.

movies_watched_df = user_movie_df[movies_watched_id]

In [ ]:
movies_watched_df.tail()

movieId,1,3,7,17,52,62,135,140,141,260,494,628,648,653,708,719,733,736,743,762,780,788,802,1073
userId,,,,,,,,,,,,,,,,,,,,,,,,
138485.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138486.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,NaN,NaN,NaN,NaN,4.5,NaN,NaN,5.0,NaN,NaN,4.5
138487.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138488.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
138490.0,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
movies_watched_df.shape

(69246, 24)

In [ ]:
# Step-2: Create a new dataframe, user_movie_count, showing how many films each user has also watched
#         from the selected user's film list.

user_movie_count = movies_watched_df.T.notnull().sum()

In [ ]:
user_movie_count = user_movie_count.reset_index()

In [ ]:
user_movie_count.columns = ["userId", "movie_count"]

In [ ]:
user_movie_count.tail(10)

,userId,movie_count
69236,138476.0,0
69237,138481.0,1
69238,138482.0,1
69239,138483.0,8
69240,138484.0,6
69241,138485.0,0
69242,138486.0,5
69243,138487.0,0
69244,138488.0,1
69245,138490.0,1


In [ ]:
# Step-3: Identify users who have watched at least 50% of the films watched by the selected user.
#         These users are considered similar to the selected user.

perc = len(movies_watched_id) * 50 / 100

In [ ]:
users_same_movies = user_movie_count[user_movie_count["movie_count"] > perc]["userId"]

In [ ]:
len(users_same_movies)

4112

### Determining the Users Most Similar to the User to Make Suggestion

In [ ]:
# Step-4: Filter movies_watched_df to retain only the Ids of similar users.

final_df = movies_watched_df[movies_watched_df.index.isin(users_same_movies)]

In [ ]:
final_df.head()

movieId,1,3,7,17,52,62,135,140,141,260,494,628,648,653,708,719,733,736,743,762,780,788,802,1073
userId,,,,,,,,,,,,,,,,,,,,,,,,
6.0,5.0,3.0,5.0,5.0,5.0,5.0,3.0,4.0,5.0,4.0,4.0,4.0,5.0,4.0,4.0,3.0,3.0,2.0,3.0,3.0,3.0,4.0,3.0,1.0
19.0,5.0,4.0,5.0,4.0,NaN,5.0,4.0,NaN,5.0,NaN,4.0,4.0,3.0,NaN,NaN,3.0,4.0,4.0,NaN,NaN,4.0,4.0,4.0,5.0
54.0,4.0,NaN,NaN,2.0,4.0,NaN,2.0,NaN,NaN,4.0,4.0,NaN,3.0,NaN,3.0,3.0,4.0,4.0,NaN,NaN,5.0,3.0,NaN,3.0
69.0,4.0,NaN,NaN,NaN,NaN,3.0,2.0,NaN,NaN,5.0,3.0,NaN,4.0,2.0,3.0,4.0,4.0,3.0,2.0,3.0,4.0,3.0,NaN,4.0
143.0,4.0,3.0,4.0,NaN,NaN,NaN,NaN,NaN,5.0,4.0,4.0,4.0,4.0,3.0,NaN,3.0,4.0,3.0,NaN,NaN,4.0,4.0,3.0,3.0


In [ ]:
final_df.shape

(4112, 24)

In [ ]:
# Step-5: Create a new dataframe, corr_df, containing correlation scores between the selected user
#         and other users based on their film ratings.

corr_df = final_df.T.corr().unstack().sort_values()

In [ ]:
corr_df = pd.DataFrame(corr_df, columns=["corr"])

In [ ]:
corr_df.head()

,,corr
userId,userId,
31080.0,62702.0,-1.0
23375.0,128356.0,-1.0
99958.0,1301.0,-1.0
62702.0,31080.0,-1.0
60705.0,128690.0,-1.0


In [ ]:
corr_df.index.names = ['user_id_1', 'user_id_2']

In [ ]:
corr_df = corr_df.reset_index()

In [ ]:
# Step-6: Create a dataframe, top_users, to store users with a high correlation (above 0.5) with the selected user.
#         Sort by correlation in descending order.

top_users = corr_df[(corr_df["user_id_1"] == target_user_id) & (corr_df["corr"] >= 0.5)][["user_id_2", "corr"]].reset_index(drop=True)

In [ ]:
top_users = top_users.sort_values(by='corr', ascending=False)

In [ ]:
top_users.rename(columns={"user_id_2": "userId"}, inplace=True)

In [ ]:
top_users

,userId,corr
240,6.0,1.000000
239,57612.0,0.835603
238,98211.0,0.832822
237,43852.0,0.823816
236,11050.0,0.804650
235,69887.0,0.786947
234,122189.0,0.785424
233,20506.0,0.771112
232,113797.0,0.763969
231,35120.0,0.758340


In [ ]:
top_users.shape

(241, 2)

In [ ]:
# Step-7: Merge the top_users dataframe with the ratings dataset to access ratings given by highly correlated users.
#         Exclude the target user from the merged data.

top_users_ratings = top_users.merge(rating_df[["userId", "movieId", "rating"]], how='inner')

In [ ]:
top_users_ratings = top_users_ratings[top_users_ratings["userId"] != target_user_id]

In [ ]:
top_users_ratings["userId"].unique()

array([ 57612.,  98211.,  43852.,  11050.,  69887., 122189.,  20506.,
       113797.,  35120.,  37056.,   6974.,  54451.,  39647.,  36843.,
        53258., 106184.,  49837.,  95993., 125978.,  37311.,  70871.,
        77695., 117274.,   3227.,  90106.,  99958.,  15332.,    143.,
        76714.,   7204.,  35253.,  73057., 120354.,  17391.,  76735.,
       135394., 104815.,  62818.,  39899.,  52075.,  43412.,  17587.,
        39931.,  16765.,  76595.,  30622.,  43386.,  52241.,  50259.,
         2902., 135531.,  38139., 137387., 119591., 112582.,  26830.,
        56674.,  93021., 112093.,   6812.,  28971.,  38492., 128224.,
        60401.,   3890., 136599.,  21540.,  94882.,  68227.,  20701.,
          158., 136897.,  78685.,  28771.,   2700.,  30901.,  48703.,
        49429.,  35091., 121814.,  59297., 108332.,  60769.,  55102.,
       118089., 122732.,  89783., 112406.,  58679.,  98875., 129970.,
        56718., 115855.,  12772., 113235.,  24113., 111214.,  79557.,
       109062., 1169

In [ ]:
top_users_ratings.head()

,userId,corr,movieId,rating
24,57612.0,0.835603,1,4.0
25,57612.0,0.835603,3,3.0
26,57612.0,0.835603,7,5.0
27,57612.0,0.835603,11,5.0
28,57612.0,0.835603,17,5.0


### Calculation of Weighted Average Recommendation Score

In [ ]:
# Step-8: Create a new column, weighted_rating, which is the product of each user's correlation score
#         and their rating for a given movie.

top_users_ratings['weighted_rating'] = top_users_ratings['corr'] * top_users_ratings['rating']

In [ ]:
top_users_ratings.head()

,userId,corr,movieId,rating,weighted_rating
24,57612.0,0.835603,1,4.0,3.342410
25,57612.0,0.835603,3,3.0,2.506808
26,57612.0,0.835603,7,5.0,4.178013
27,57612.0,0.835603,11,5.0,4.178013
28,57612.0,0.835603,17,5.0,4.178013


In [ ]:
# Step-9: Create a new dataframe, recommendation_df, with movie ids and the average weighted ratings
#         across all similar users for each movie.

recommendation_df = top_users_ratings.groupby('movieId').agg({"weighted_rating": "mean"})

In [ ]:
recommendation_df = recommendation_df.reset_index()

In [ ]:
recommendation_df.sort_values("weighted_rating", ascending=False).head()

,movieId,weighted_rating
2843,3446,3.855560
7244,33826,3.636264
4556,5681,3.636264
3428,4183,3.590262
7471,41714,3.571325


In [ ]:
# Step-10: Filter movies with an average weighted rating above 3 and that are not watched by the selected user.
#          Sort them by weighted rating in descending order to get the best recommendations.

recommendation_df = recommendation_df[(recommendation_df["weighted_rating"] > 3) & (~recommendation_df['movieId'].isin(movies_watched_id))].sort_values("weighted_rating", ascending=False)

recommendation_df.head()

,movieId,weighted_rating
2843,3446,3.855560
4556,5681,3.636264
7244,33826,3.636264
3428,4183,3.590262
6662,26501,3.571325


In [ ]:
# Step-11: Select the top 5 movies with the highest average weighted ratings as recommendations.

corr_recommendations = recommendation_df.sort_values("weighted_rating", ascending=False)[:5]
print(corr_recommendations)

      movieId  weighted_rating
2843     3446         3.855560
7244    33826         3.636264
4556     5681         3.636264
3428     4183         3.590262
8479    61373         3.571325


In [ ]:
corr_movieId = corr_recommendations['movieId'].tolist()
print(corr_movieId)

[3446, 33826, 5681, 4183, 61373]


In [ ]:
# Get the corresponding movie title & genre of the recommendations

convert_id_to_title_and_genre(corr_movieId)

Movie: Funny Bones (1995), Genre: Comedy|Drama
Movie: Saint Ralph (2004), Genre: Comedy|Drama
Movie: Fidel (2001), Genre: Documentary
Movie: Unbelievable Truth, The (1989), Genre: Comedy|Drama
Movie: Woman in Black, The (1989), Genre: Horror|Mystery


## 2.2 Matrix Factorization Method - Singular Value Decomposition

In [ ]:
# Step 1: Define the reader to specify the rating scale

reader = Reader(rating_scale=(0.5, 5.0))

# Step 2: Load the dataset into Surprise format

data = Dataset.load_from_df(common_movies[['userId', 'movieId', 'rating']], reader)

# Step 3: Split the data into training and testing sets

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Step 4: Initialize the SVD model

svd_model = SVD()

# Step 5: Train the model on the training set

svd_model.fit(trainset)

In [ ]:
# Step 6: Predict ratings for the test set and calculate RMSE to evaluate the model

predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions)

RMSE: 0.7873


In [ ]:
# Step 7: Define a function to get recommendations for a specific user

def get_user_recommendations(user_id, top_n=5):
    # Get all movies rated by the user
    user_rated_movies = common_movies[common_movies['userId'] == user_id]['movieId'].tolist()

    # Predict ratings for all movies the user hasn't rated yet
    movie_predictions = [
        (movie_id, svd_model.predict(user_id, movie_id).est)
        for movie_id in common_movies['movieId'].unique()
        if movie_id not in user_rated_movies
    ]

    # Sort predictions by estimated rating in descending order and select the top N
    movie_predictions.sort(key=lambda x: x[1], reverse=True)
    top_movies = movie_predictions[:top_n]

    return top_movies

In [ ]:
# Get recommendations for a specific user

svd_recommendations = get_user_recommendations(target_user_id)
svd_movieId = [i[0] for i in svd_recommendations]
print(svd_movieId)

[92259, 318, 110, 26614, 1221]


In [ ]:
# Get the corresponding movie title & genre of the recommendations

convert_id_to_title_and_genre(svd_movieId)

Movie: Intouchables (2011), Genre: Comedy|Drama
Movie: Shawshank Redemption, The (1994), Genre: Crime|Drama
Movie: Braveheart (1995), Genre: Action|Drama|War
Movie: Bourne Identity, The (1988), Genre: Action|Adventure|Drama|Mystery|Thriller
Movie: Godfather: Part II, The (1974), Genre: Crime|Drama


# Hybrid Recommendation

In [ ]:
# Step 1: Filter ratings for movie IDs from all methods separately

cbf_filtered_ratings = rating_df[rating_df['movieId'].isin(cbf_movieId)]
corr_filtered_ratings = rating_df[rating_df['movieId'].isin(corr_movieId)]
svd_filtered_ratings = rating_df[rating_df['movieId'].isin(svd_movieId)]

In [ ]:
svd_filtered_ratings.head()

,userId,movieId,rating,timestamp
12,1,318,4.0,2005-04-02 23:33:18
178,2,110,4.0,2000-11-21 15:30:58
247,3,318,5.0,1999-12-11 13:09:26
299,3,1221,5.0,1999-12-11 13:01:34
457,5,110,4.0,1996-12-25 15:26:09


In [ ]:
# Step 2: Calculate average ratings of these movies for their corresponding dataframes

cbf_average_ratings = cbf_filtered_ratings.groupby('movieId')['rating'].mean().reset_index()
corr_average_ratings = corr_filtered_ratings.groupby('movieId')['rating'].mean().reset_index()
svd_average_ratings = svd_filtered_ratings.groupby('movieId')['rating'].mean().reset_index()

In [ ]:
svd_average_ratings.head()

,movieId,rating
0,110,4.042534
1,318,4.446990
2,1221,4.275641
3,26614,4.031519
4,92259,4.132396


In [ ]:
# Step 3: Sort by average rating and get the 2 highest rated movies from each method

cbf_top_2 = cbf_average_ratings.sort_values(by='rating', ascending=False)[:2]
corr_top_2 = corr_average_ratings.sort_values(by='rating', ascending=False)[:2]
svd_top_2 = svd_average_ratings.sort_values(by='rating', ascending=False)[:2]

In [ ]:
cbf_top_2

,movieId,rating
3,103606,3.675258
2,103539,3.544601


In [ ]:
corr_top_2

,movieId,rating
1,4183,3.854508
3,33826,3.601604


In [ ]:
svd_top_2

,movieId,rating
1,318,4.446990
2,1221,4.275641


In [ ]:
# Make a combined list of all these movie IDs

cbf_top_2_ids = cbf_top_2['movieId'].values.tolist()
corr_top_2_ids = corr_top_2['movieId'].values.tolist()
svd_top_2_ids = svd_top_2['movieId'].values.tolist()
final_recommendations_id = list(set(cbf_top_2_ids + corr_top_2_ids + svd_top_2_ids))
final_recommendations_id

[33826, 1221, 103539, 103606, 4183, 318]

In [ ]:
# Get the corresponding movie title & genre of the recommendations

convert_id_to_title_and_genre(final_recommendations_id)

Movie: Saint Ralph (2004), Genre: Comedy|Drama
Movie: Godfather: Part II, The (1974), Genre: Crime|Drama
Movie: The Spectacular Now (2013), Genre: Comedy|Drama|Romance
Movie: Stuck in Love (2012), Genre: Comedy|Drama|Romance
Movie: Unbelievable Truth, The (1989), Genre: Comedy|Drama
Movie: Shawshank Redemption, The (1994), Genre: Crime|Drama
